# PagedAttention：CPU 块池模拟

此 notebook 只用整数模拟 KV block 的分配、回收与共享前缀；它不是 Attention kernel，也不测量显存或 GPU 速度。

In [ ]:
class BlockPool:
    def __init__(self, count=8, block_size=4):
        self.block_size = block_size
        self.free = list(range(count))
        self.tables = {}
        self.refs = {}

    def allocate(self, request):
        if not self.free:
            raise MemoryError('没有空闲 KV block')
        block = self.free.pop(0)
        self.tables.setdefault(request, []).append(block)
        self.refs[block] = self.refs.get(block, 0) + 1
        return block

    def share_prefix(self, request, blocks):
        self.tables[request] = list(blocks)
        for block in blocks:
            self.refs[block] += 1

    def release(self, request):
        for block in self.tables.pop(request, []):
            self.refs[block] -= 1
            if self.refs[block] == 0:
                del self.refs[block]
                self.free.append(block)
        self.free.sort()

    def show(self):
        print('block tables:', self.tables)
        print('引用计数:', self.refs, '空闲:', self.free)

In [ ]:
pool = BlockPool(count=6, block_size=4)
# A 的 6 token 需要两个逻辑块；它们在物理上可以不连续。
pool.allocate('A'); pool.allocate('A')
pool.allocate('B')
pool.show()

# B 完成后，C 可立即拿到 B 归还的物理块。
pool.release('B')
pool.allocate('C')
pool.show()

In [ ]:
# D 与 A 具有完整、相同的可缓存前缀，因此共享 A 的两个块。
pool.share_prefix('D', pool.tables['A'])
pool.allocate('D')  # D 不同的后缀需要新块
pool.show()

pool.release('A')
print('释放 A 后，共享块仍被 D 引用：')
pool.show()
pool.release('D')
print('释放 D 后，共享块才可回收：')
pool.show()

思考：把 `count` 改小直到触发 `MemoryError`。真实运行时会由调度器、KV cache manager 和块池共同决定是否准入，而不会把这个玩具异常直接暴露给用户。